<a href="https://colab.research.google.com/github/cjkcaya/DELIST/blob/main/Gcollab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Oct 15 21:04:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/lumantasamueljunard/sto-nino-threat-evacuation.git
%cd sto-nino-threat-evacuation

Cloning into 'sto-nino-threat-evacuation'...
fatal: could not read Username for 'https://github.com': No such device or address
[Errno 2] No such file or directory: 'sto-nino-threat-evacuation'
/content


In [ ]:
import dynamic_evacuation as de
print('Rooms:', len(de.ROOMS), '| Devices:', len(de.DEVICES))

ModuleNotFoundError: No module named 'dynamic_evacuation'

In [ ]:
!pip install -q ultralytics librosa

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests, io

# Download a standard demo image used by Ultralytics
url = 'https://ultralytics.com/images/bus.jpg'
img = Image.open(io.BytesIO(requests.get(url).content))

model = YOLO('yolov8n.pt')       # downloads ~6 MB, runs in seconds
results = model(img)
print(results[0].boxes.cls.tolist())     # class IDs detected
print(results[0].names)                   # id -> class-name mapping

In [ ]:
import librosa, numpy as np

# Generate a 3-second 440 Hz tone in memory — no file needed
sr = 22050
t = np.linspace(0, 3, 3 * sr, endpoint=False)
y = 0.5 * np.sin(2 * np.pi * 440 * t)

rms = float(np.sqrt(np.mean(y ** 2)))
centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
print(f'RMS energy: {rms:.3f}')
print(f'Spectral centroid: {centroid:.1f} Hz')

In [ ]:
!pip install -q ultralytics huggingface_hub

In [ ]:
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id="Subh775/Firearm_Detection_Yolov8n",
    filename="weights/best.pt"
)
print("Model downloaded to:", model_path)

In [ ]:
from ultralytics import YOLO
from PIL import Image
import requests, io

# Load the downloaded model
model = YOLO(model_path)

# Download a standard demo image used by Ultralytics
url = 'https://ultralytics.com/images/bus.jpg'
img = Image.open(io.BytesIO(requests.get(url).content))

# Run inference
results = model(img)

# Print detected class IDs and class names
print("Detected class IDs:", results[0].boxes.cls.tolist())
print("Class names mapping:", results[0].names)

In [ ]:
from google.colab import files
print("Upload your test_video.mp4 (and optionally test_audio.wav). You can select both at once.")
uploaded = files.upload()
for name, data in uploaded.items():
    print(f"Saved: {name}  ({len(data)/1024:.1f} KB)")

In [ ]:
import os
print(os.listdir())

In [ ]:
from ultralytics import YOLO

firearm_model = YOLO(model_path)              # from Step 2 earlier
pose_model    = YOLO('yolov8n-pose.pt')       # auto-downloads ~7 MB

print("Firearm model classes:", firearm_model.names)
print("Pose model classes:", pose_model.names)

In [ ]:
import cv2
import numpy as np

def firearm_object_score(video_path, model,
                         frame_stride=5,
                         confidence=0.35,
                         max_frames=None):
    """Return (score_0_to_100, stats_dict)."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    frames_seen = 0
    frames_with_gun = 0
    max_confidence = 0.0
    all_detections = []

    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % frame_stride == 0:
            result = model(frame, conf=confidence, verbose=False)[0]
            boxes = result.boxes
            frames_seen += 1
            if boxes is not None and len(boxes) > 0:
                frames_with_gun += 1
                confs = boxes.conf.tolist()
                max_confidence = max(max_confidence, max(confs))
                all_detections.append({
                    'frame': i,
                    'n_boxes': len(boxes),
                    'max_conf': max(confs),
                })
        i += 1
        if max_frames and frames_seen >= max_frames:
            break

    cap.release()

    if frames_seen == 0:
        return 0.0, {'frames_seen': 0, 'frames_with_gun': 0}

    score = 100.0 * frames_with_gun / frames_seen
    stats = {
        'fps': fps,
        'total_frames': total_frames,
        'frames_analyzed': frames_seen,
        'frames_with_gun': frames_with_gun,
        'max_confidence': round(max_confidence, 3),
        'detections': all_detections[:20],  # cap for readability
    }
    return round(score, 2), stats


# --- Run it on your uploaded video ---
video_file = ' Glock 18C Fast Mode! 💨💨💨.mp4'
object_score, obj_stats = firearm_object_score(video_file, firearm_model)

print(f"Object score: {object_score} / 100")
print(f"Frames analyzed: {obj_stats['frames_analyzed']}")
print(f"Frames with a gun: {obj_stats['frames_with_gun']}")
print(f"Highest confidence: {obj_stats['max_confidence']}")


In [ ]:
import os
print("Files in current directory:")
for f in sorted(os.listdir()):
    if not f.startswith('.'):
        size = os.path.getsize(f)
        print(f"  {f}  ({size/1024:.1f} KB)")

In [ ]:
import os, shutil

# Find any .mp4 in the current folder
candidates = [f for f in os.listdir('.') if f.lower().endswith('.mp4')]
print("Found mp4 files:", candidates)

# If the Glock video is there, rename it
for f in candidates:
    if 'Glock' in f or 'glock' in f:
        if os.path.exists('test_video.mp4'):
            os.remove('test_video.mp4')
        shutil.move(f, 'test_video.mp4')
        print(f"Renamed '{f}' -> 'test_video.mp4'")
        break

# Confirm
print("\nCurrent files:")
for f in sorted(os.listdir('.')):
    if not f.startswith('.'):
        print(" ", f)

In [ ]:
import cv2
cap = cv2.VideoCapture('test_video.mp4')
print("Opened:", cap.isOpened())
print("Frames:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))
ok, frame = cap.read()
print("First frame read:", ok)
if ok:
    print("Frame shape:", frame.shape)
cap.release()

In [ ]:
video_file = 'test_video.mp4'   # or 'test_video_h264.mp4' if re-encoded

object_score, obj_stats = firearm_object_score(video_file, firearm_model)

print(f"Object score: {object_score} / 100")
print(f"Video opened: {obj_stats['opened']}")
print(f"Frames analyzed: {obj_stats['frames_analyzed']}")
print(f"Frames with a gun: {obj_stats['frames_with_gun']}")
print(f"Highest confidence: {obj_stats['max_confidence']}")


In [ ]:
import cv2

def firearm_object_score(video_path, model,
                         frame_stride=5,
                         confidence=0.35,
                         max_frames=None):
    """Return (score_0_to_100, stats_dict). Stats always has the same keys."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    frames_seen = 0
    frames_with_gun = 0
    max_confidence = 0.0
    all_detections = []

    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % frame_stride == 0:
            result = model(frame, conf=confidence, verbose=False)[0]
            boxes = result.boxes
            frames_seen += 1
            if boxes is not None and len(boxes) > 0:
                frames_with_gun += 1
                confs = boxes.conf.tolist()
                max_confidence = max(max_confidence, max(confs))
                all_detections.append({
                    'frame': i,
                    'n_boxes': len(boxes),
                    'max_conf': round(max(confs), 3),
                })
        i += 1
        if max_frames and frames_seen >= max_frames:
            break

    cap.release()

    score = 100.0 * frames_with_gun / frames_seen if frames_seen > 0 else 0.0
    stats = {
        'opened': total_frames > 0,
        'fps': fps,
        'total_frames': total_frames,
        'frames_analyzed': frames_seen,
        'frames_with_gun': frames_with_gun,
        'max_confidence': round(max_confidence, 3),
        'detections': all_detections[:20],
    }
    return round(score, 2), stats

In [ ]:
def pose_score_from_video(video_path, model,
                          frame_stride=5,
                          confidence=0.35):
    cap = cv2.VideoCapture(video_path)
    frames_seen = 0
    frames_with_person = 0
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % frame_stride == 0:
            result = model(frame, conf=confidence, verbose=False)[0]
            frames_seen += 1
            if result.keypoints is not None and len(result.keypoints) > 0:
                frames_with_person += 1
        i += 1
    cap.release()
    return round(100.0 * frames_with_person / frames_seen, 2) if frames_seen else 0.0


pose_score = pose_score_from_video(video_file, pose_model)
print(f"Pose score: {pose_score} / 100")

In [ ]:
!ffmpeg -y -i {video_file} -vn -acodec pcm_s16le -ar 22050 -ac 1 extracted_audio.wav 2>/dev/null

import os, librosa, numpy as np

has_audio = os.path.exists('extracted_audio.wav') and os.path.getsize('extracted_audio.wav') > 1000

if has_audio:
    y, sr = librosa.load('extracted_audio.wav', sr=22050, mono=True)
    rms = float(np.sqrt(np.mean(y ** 2)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    loud  = min(rms * 400, 1.0)
    sharp = min(centroid / 4000.0, 1.0)
    noisy = min(zcr * 8, 1.0)
    acoustic_score = round(100.0 * (0.5*loud + 0.3*sharp + 0.2*noisy), 2)
    print(f"Acoustic score: {acoustic_score} / 100")
    print({'rms': round(rms,4), 'centroid_hz': round(centroid,1), 'zcr': round(zcr,4)})
else:
    acoustic_score = 0.0
    print("No audio track — acoustic score set to 0.")

In [ ]:
import importlib, dynamic_evacuation as de
importlib.reload(de)

DEVICE_ID = 'B1-F1-CAM-L'
EVENT_ID  = 'colab-run-1'

scores = {'acoustic': acoustic_score, 'pose': pose_score, 'object': object_score}
print("Scores:", scores)

c = de.classify_threat(scores['acoustic'], scores['pose'], scores['object'])
print("Classification:", c)

obs = de.Observation(event_id=EVENT_ID, device_id=DEVICE_ID,
                     level=c['level'], count=1, incident_id='colab-inc-1')
snap = de.assess([obs])

print(f"\nRaw detections: {snap['raw_detection_count']}")
print(f"Possible threat locations: {snap['possible_threat_locations']}")

for room in ['B1-F1-R1', 'B1-F4-R3', 'B2-F4-R2']:
    print(de.format_route(de.route(room, snap)))

In [ ]:
import json

# Repackage the results from your run into a portable record
detection_record = {
    "source": "colab_real_detection",
    "video_file": video_file,
    "device_id": DEVICE_ID,
    "event_id": EVENT_ID,
    "incident_id": "colab-inc-1",
    "scores": {
        "acoustic": acoustic_score,
        "pose": pose_score,
        "object": object_score,
    },
    "object_stats": {
        "frames_analyzed": obj_stats["frames_analyzed"],
        "frames_with_gun": obj_stats["frames_with_gun"],
        "max_confidence": obj_stats["max_confidence"],
    },
    "classification": c,   # the dict returned by classify_threat
}

with open("detections.json", "w") as f:
    json.dump([detection_record], f, indent=2)

print("Saved detections.json")
print(json.dumps(detection_record, indent=2))

# Trigger a browser download
from google.colab import files
files.download("detections.json")